# Frozen ADMET results

Read the five saved endpoint CSVs; no embeddings or models are loaded. Run all cells in the main environment. For another reproduction, copy the notebook into its result directory and run it from there. All checked-in results use the uniform grouped-holdout rerun adopted on 2026-09-23; see README.md for provenance and the logistic-convergence warning.

Scores are already computed on individual held-out spectrum records (not molecule-averaged predictions). Ames uses ROC-AUC: ranking positives above negatives from predicted probabilities. Regression uses R² = 1 − squared prediction error / squared deviation from the test mean; negative values are possible. Average precision, MAE and other metrics remain in the input CSVs; this notebook does not recompute predictions or average ROC-AUC with R².

Each mean rank gives equal weight to the five endpoints; lower is better, ties use average ranks. Pools are ranked independently. Single-run ranks are descriptive, not significance tests. ADMET exclusion from rich pretraining does not establish absence from SimNMR/NMRGym or upstream pretraining corpora.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

DATASETS = ["solubility_aqsoldb", "ld50_zhu", "lipophilicity_astrazeneca", "sangster_logp", "ames"]
RESULTS_DIR = Path.cwd()
if not all((RESULTS_DIR / f"{dataset}.csv").is_file() for dataset in DATASETS):
    RESULTS_DIR = RESULTS_DIR / "results/property_prediction"
ANALYSIS_DIR = RESULTS_DIR / "analysis"
BASELINES = ["morgan", "nmrpeak", "nmrsolver", "nmrtrans", "ultranmr", "unimol2"]
COMBINED = ["unimol2+ultranmr", "unimol2+nmrpeak", "unimol2+nmrsolver",
            "morgan+nmrpeak", "morgan+ultranmr", "morgan+nmrsolver"]
FOMONMR = ["fomonmr-pre-shifts", "fomonmr-post-shifts", "fomonmr-post-rich", "fomonmr-post-unimol-rich"]
REPRESENTATIONS = BASELINES + COMBINED + FOMONMR + ["morgan+fomonmr-post-unimol-rich"]
MAIN_STANDALONE = BASELINES + ["fomonmr-post-rich", "fomonmr-post-unimol-rich"]
MAIN_REPRESENTATIONS = MAIN_STANDALONE + COMBINED + ["morgan+fomonmr-post-unimol-rich"]


## Load and check completeness

Require every endpoint, representation and probe, finite primary scores, and identical cohort sizes within each endpoint. Equal counts alone do not prove equal IDs: the probe runner constructs the common ID intersection.

In [ ]:
frames = []
for dataset in DATASETS:
    frame = pd.read_csv(RESULTS_DIR / f"{dataset}.csv")
    assert frame["dataset"].eq(dataset).all(), f"Wrong dataset in {dataset}.csv"
    frames.append(frame)
results = pd.concat(frames, ignore_index=True)
keys = ["dataset", "representation", "probe"]
assert not results.duplicated(keys).any(), "Duplicate result rows"
expected = {(d, r, p) for d in DATASETS for r in REPRESENTATIONS for p in ("linear", "mlp")}
assert set(results[keys].itertuples(index=False, name=None)) == expected, "Missing or unexpected results"
counts = ["n_train", "n_test", "n_train_molecules", "n_test_molecules"]
assert results.groupby("dataset")[counts].nunique().eq(1).all().all(), "Different cohorts"
results["metric"] = np.where(results["dataset"].eq("ames"), "roc_auc", "r2")
results["score"] = np.where(results["dataset"].eq("ames"), results["roc_auc"], results["r2"])
assert np.isfinite(results["score"]).all(), "Non-finite primary metrics"
scores = results[keys + ["metric", "score"] + counts].sort_values(keys)
display(scores.groupby("dataset")[counts].first().reindex(DATASETS))


## Endpoint scores and comparison pools

The context note compares eight standalone representations and fifteen main methods including concatenations. Both pools retain the two rich FoMoNMR posttraining checkpoints. Shift-only conditions remain in the full seventeen-method pool and the FoMoNMR diagnostic comparison; they are not selected per task. Published-only pools are retained for reference. Fine-tuning is separate.


In [ ]:
pools = {
    "main_standalone_8": MAIN_STANDALONE,
    "main_15": MAIN_REPRESENTATIONS,
    "all_17": REPRESENTATIONS,
    "published_12": BASELINES + COMBINED,
    "published_standalone_6": BASELINES,
    "nmr_only_8": [r for r in BASELINES if r not in ("morgan", "unimol2")] + FOMONMR,
    "fomonmr_4": FOMONMR,
    "morgan_comparisons": ["morgan"] + [r for r in REPRESENTATIONS if r.startswith("morgan+")],
    "unimol2_comparisons": ["unimol2"] + [r for r in COMBINED if r.startswith("unimol2+")],
}
rank_frames = []
for pool, representations in pools.items():
    subset = scores[scores["representation"].isin(representations)].copy()
    subset["rank"] = subset.groupby(["dataset", "probe"])["score"].rank(ascending=False, method="average")
    summary = subset.groupby(["probe", "representation"], as_index=False).agg(
        mean_rank=("rank", "mean"), n_endpoints=("dataset", "nunique"))
    assert summary["n_endpoints"].eq(len(DATASETS)).all()
    summary.insert(0, "pool", pool)
    rank_frames.append(summary)
mean_rank = pd.concat(rank_frames, ignore_index=True).sort_values(["pool", "probe", "mean_rank", "representation"])

for probe in ("linear", "mlp"):
    print(probe.upper(), "— Ames: ROC-AUC; other endpoints: R²")
    display(scores[scores["probe"].eq(probe)].pivot(
        index="representation", columns="dataset", values="score").reindex(index=REPRESENTATIONS, columns=DATASETS).round(4))

for probe in ("linear", "mlp"):
    print(probe.upper(), "— main standalone comparison")
    table = scores[scores["probe"].eq(probe)].pivot(index="representation", columns="dataset", values="score").reindex(index=MAIN_STANDALONE, columns=DATASETS)
    ranks = mean_rank[mean_rank["pool"].eq("main_standalone_8") & mean_rank["probe"].eq(probe)].set_index("representation")["mean_rank"]
    display(table.join(ranks).sort_values("mean_rank").round(4))


In [ ]:
for pool in ("fomonmr_4", "main_standalone_8", "main_15"):
    print(pool, "— mean rank (lower is better)")
    display(mean_rank[mean_rank["pool"].eq(pool)].pivot(
        index="representation", columns="probe", values="mean_rank").sort_values("linear").round(2))


## Complementarity over the corresponding structure-only baseline

Compare every measured concatenation with its molecular baseline on each endpoint. The best-combination tables match the context note. Their winners are selected retrospectively using test scores: this is a descriptive, optimistic comparison, not a method selected without access to the test set. No cross-task average of ROC-AUC and R² differences is used.


In [ ]:
comparisons = scores[scores["representation"].str.contains("+", regex=False)][["dataset", "probe", "representation", "metric", "score"]].copy()
comparisons["baseline"] = comparisons["representation"].str.split("+", regex=False).str[0]
baselines = scores[scores["representation"].isin(["morgan", "unimol2"])][["dataset", "probe", "representation", "score"]].rename(columns={"representation": "baseline", "score": "baseline_score"})
comparisons = comparisons.merge(baselines, on=["dataset", "probe", "baseline"], validate="many_to_one")
comparisons["delta"] = comparisons["score"] - comparisons["baseline_score"]
comparisons["improved"] = comparisons["delta"] > 0
comparisons = comparisons.sort_values(["probe", "baseline", "representation", "dataset"])
for probe in ("linear", "mlp"):
    print(probe, "— per-endpoint differences from the structure-only baseline")
    display(comparisons[comparisons["probe"].eq(probe)].pivot(index="representation", columns="dataset", values="delta").reindex(columns=DATASETS).round(4))

for probe in ("linear", "mlp"):
    print(probe.upper(), "— all molecular baselines and concatenations")
    members = ["morgan", "unimol2"] + [r for r in REPRESENTATIONS if "+" in r]
    display(scores[scores["probe"].eq(probe)].pivot(index="representation", columns="dataset", values="score").reindex(index=members, columns=DATASETS).round(4))

best = comparisons.loc[comparisons.groupby(["baseline", "probe", "dataset"])["score"].idxmax()]
for baseline in ("morgan", "unimol2"):
    print(baseline, "— best measured combination by endpoint (test-selected)")
    display(best[best["baseline"].eq(baseline)][["probe", "dataset", "baseline_score", "representation", "score", "delta"]].round(4))


## Save the three derived tables

Interpretation belongs in contex/Property Prediction.md. These exports can be regenerated and are ignored by Git.

In [ ]:
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
scores.to_csv(ANALYSIS_DIR / "scores.csv", index=False)
mean_rank.to_csv(ANALYSIS_DIR / "mean_rank.csv", index=False)
comparisons.to_csv(ANALYSIS_DIR / "complementarity.csv", index=False)
print(f"Saved scores.csv, mean_rank.csv and complementarity.csv to {ANALYSIS_DIR}")
